# Warstwa 5: Wizualizacje — Dashboard analityczny

**Cel:** Interaktywne wykresy Plotly odpowiadające na kluczowe pytania biznesowe  
**Wejście:** `data/warehouse/youtube_dw.duckdb`

### Pytania analityczne:
1. Które kategorie dominują w trendach?
2. Jak różnią się trendy w różnych krajach?
3. Kiedy najlepiej publikować (godzina i dzień tygodnia)?
4. Jaki jest rozkład wyświetleń?
5. Jak długość wideo wpływa na popularność?
6. Jakie jest zaangażowanie (lajki) per kategoria?


In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DB_PATH = Path("../data/warehouse/youtube_dw.duckdb")
con = duckdb.connect(str(DB_PATH), read_only=True)
print("Połączono z hurtownią")

Połączono z hurtownią


In [2]:
# ── Wykres 1: Top kategorie wg liczby filmów i wyświetleń ────────────────────
df_cat = con.execute("""
SELECT c.category_name,
       COUNT(*)                           AS filmy,
       ROUND(SUM(f.view_count) / 1e9, 2) AS mld_wysw,
       ROUND(AVG(f.like_ratio) * 100, 3) AS avg_like_pct
FROM fact_videos f
JOIN dim_category c USING (category_id)
GROUP BY c.category_name
ORDER BY filmy DESC
""").fetchdf()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Liczba filmów w trendach", "Łączne wyświetlenia (mld)"])

fig.add_trace(go.Bar(
    x=df_cat["filmy"], y=df_cat["category_name"],
    orientation="h", marker_color="#4285F4", name="Filmy"
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_cat["mld_wysw"], y=df_cat["category_name"],
    orientation="h", marker_color="#EA4335", name="Wyświetlenia"
), row=1, col=2)

fig.update_layout(
    title_text="Kategorie YouTube — liczba filmów vs wyświetlenia",
    height=500, showlegend=False
)
fig.show()

In [3]:
# ── Wykres 2: Rozkład wyświetleń per region (box plot) ───────────────────────
df_reg = con.execute("""
SELECT r.region_name, f.view_count / 1e6 AS mln_wysw, f.view_category
FROM fact_videos f
JOIN dim_region r ON f.region = r.region_code
WHERE f.view_count > 0
""").fetchdf()

fig = px.box(
    df_reg, x="region_name", y="mln_wysw",
    color="region_name",
    title="Rozkład wyświetleń (mln) per kraj",
    labels={"region_name": "Kraj", "mln_wysw": "Wyświetlenia (mln)"},
    log_y=True
)
fig.update_layout(showlegend=False)
fig.show()

In [4]:
# ── Wykres 3: Heatmapa — godzina × dzień tygodnia (wyświetlenia) ─────────────
df_heat = con.execute("""
SELECT publish_hour, publish_dow_name, publish_dow,
       ROUND(AVG(view_count) / 1e6, 2) AS avg_mln
FROM fact_videos
GROUP BY publish_hour, publish_dow_name, publish_dow
ORDER BY publish_dow, publish_hour
""").fetchdf()

pivot = df_heat.pivot_table(
    index="publish_dow_name", columns="publish_hour",
    values="avg_mln", fill_value=0
)
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
pivot = pivot.reindex([d for d in day_order if d in pivot.index])

fig = px.imshow(
    pivot,
    color_continuous_scale="YlOrRd",
    title="Średnie wyświetlenia (mln) wg dnia tygodnia i godziny publikacji",
    labels={"x": "Godzina", "y": "Dzień tygodnia", "color": "mln wysw"}
)
fig.update_layout(height=350)
fig.show()

In [5]:
# ── Wykres 4: Udział kategorii viralności ────────────────────────────────────
df_pie = con.execute("""
SELECT view_category, COUNT(*) AS filmy
FROM fact_videos
GROUP BY view_category
""").fetchdf()

color_map = {"low": "#BDBDBD", "medium": "#4285F4",
             "high": "#FBBC05", "viral": "#EA4335"}

fig = px.pie(
    df_pie, names="view_category", values="filmy",
    title="Rozkład kategorii popularności filmów",
    color="view_category", color_discrete_map=color_map,
    hole=0.35
)
fig.show()

In [6]:
# ── Wykres 5: Długość wideo vs wyświetlenia ──────────────────────────────────
df_dur = con.execute("""
SELECT duration_minutes, view_count / 1e6 AS mln_wysw,
       category_name, region
FROM fact_videos f
JOIN dim_category c USING (category_id)
WHERE duration_seconds BETWEEN 30 AND 3600
  AND view_count > 0
""").fetchdf()

fig = px.scatter(
    df_dur.sample(min(2000, len(df_dur)), random_state=42),
    x="duration_minutes", y="mln_wysw",
    color="category_name",
    opacity=0.6,
    title="Długość wideo vs wyświetlenia",
    labels={"duration_minutes": "Długość (min)", "mln_wysw": "Wyświetlenia (mln)"},
    log_y=True
)
fig.update_layout(height=500)
fig.show()

In [7]:
# ── Wykres 6: Zaangażowanie (like ratio) per kategoria ───────────────────────
df_eng = con.execute("""
SELECT c.category_name,
       ROUND(AVG(f.like_ratio)    * 100, 3) AS avg_like_pct,
       ROUND(AVG(f.comment_ratio) * 100, 4) AS avg_comment_pct,
       COUNT(*) AS filmy
FROM fact_videos f
JOIN dim_category c USING (category_id)
WHERE f.view_count > 50000
GROUP BY c.category_name
ORDER BY avg_like_pct DESC
""").fetchdf()

fig = px.scatter(
    df_eng,
    x="avg_like_pct", y="avg_comment_pct",
    size="filmy", text="category_name",
    title="Zaangażowanie: % lajków vs % komentarzy per kategoria",
    labels={"avg_like_pct": "Avg % lajków", "avg_comment_pct": "Avg % komentarzy"},
    color="avg_like_pct", color_continuous_scale="Viridis"
)
fig.update_traces(textposition="top center")
fig.update_layout(height=550, showlegend=False)
fig.show()

In [8]:
# ── Wykres 7: Top 15 kanałów ─────────────────────────────────────────────────
df_ch = con.execute("""
SELECT ch.channel_title,
       COUNT(*) AS filmy,
       ROUND(SUM(f.view_count) / 1e6, 1) AS laczne_mln
FROM fact_videos f
JOIN dim_channel ch USING (channel_id)
GROUP BY ch.channel_title
ORDER BY laczne_mln DESC
LIMIT 15
""").fetchdf()

fig = px.bar(
    df_ch.sort_values("laczne_mln"),
    x="laczne_mln", y="channel_title",
    orientation="h",
    color="filmy", color_continuous_scale="Blues",
    title="Top 15 kanałów — łączne wyświetlenia (mln)",
    labels={"laczne_mln": "Wyświetlenia (mln)", "channel_title": ""},
    text="laczne_mln"
)
fig.update_traces(textposition="outside")
fig.update_layout(height=500)
fig.show()

con.close()
print("Wszystkie wykresy wygenerowane!")

Wszystkie wykresy wygenerowane!
